# Main Fig 4 — Iso-compute: vs-Total + Pareto

**Layout**: N_TASKS rows × 2 cols  
Left col = AUROC vs total compute (L×K) with legend  
Right col = Pareto-optimal frontier (text annotations, no legend)

Each panel is FULL_W/2 = 3.5 in wide. Increase ROW_H for taller panels.

In [ ]:
%matplotlib inline

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    """Walk up from CWD until we find a directory containing final_results/."""
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    # Explicit fallback (edit this if auto-detect fails)
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
FINAL_RESULTS  = WORKSPACE_ROOT / "final_results"
PAPER_FIGURES  = NSRR_TOOLS / "results" / "paper_figures"
FINAL_OUT      = PAPER_FIGURES / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)

# Add utils to path (notebooks/utils/)
_nb_dir = PAPER_FIGURES / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.style import (
    apply_tbme_style, save_figure, FULL_W, HALF_W,
    MAIN_TASKS, SUPP_TASKS, ALL_TASKS, BINARY_MAIN,
    HEAD_STYLE, TASK_LABEL, FONT_ANNOT, FONT_BASE, FONT_LABEL,
)
from utils.data import set_root, load_analysis, load_heatmap, load_parquets
from utils import panels

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

set_root(WORKSPACE_ROOT)
apply_tbme_style()

# ── Confirm the workspace root is correct ─────────────────────────────────────
_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND — edit _find_workspace() fallback'}")

In [ ]:
# Panel labeling helper
def add_panel_label(ax, label, x=0.02, y=0.97):
    ax.text(x, y, label, transform=ax.transAxes,
            fontsize=8, fontweight="bold", va="top", fontfamily="serif")

In [ ]:
HEAD    = "transformer"
TASKS   = MAIN_TASKS
METRIC  = "auroc"
ROW_H   = 2.2   # inches per task row — increase for larger panels

hmaps = {t: load_heatmap("phase0_v3", t, HEAD) for t in TASKS}
print("Loaded:", {t: len(v) for t, v in hmaps.items()})

In [ ]:
fig, axes = plt.subplots(len(TASKS), 2, figsize=(FULL_W, len(TASKS) * ROW_H))

for row, task in enumerate(TASKS):
    ax_left  = axes[row, 0]   # vs_total
    ax_right = axes[row, 1]   # pareto

    panels.vs_total_panel(ax_left,  hmaps[task], col=METRIC)
    panels.pareto_panel  (ax_right, hmaps[task], col=METRIC)

    # Title only on the left panel
    ax_left.set_title(TASK_LABEL[task], fontsize=8)

    # Panel labels: a/b for row 0, c/d for row 1, …
    add_panel_label(ax_left,  f"({chr(97 + row * 2)})")
    add_panel_label(ax_right, f"({chr(97 + row * 2 + 1)})")

    # Legend: keep on left (vs_total) for first row only; suppress on pareto always
    if row > 0 and ax_left.get_legend():
        ax_left.get_legend().remove()
    if ax_right.get_legend():
        ax_right.get_legend().remove()

fig.tight_layout(h_pad=1.2, w_pad=1.0)
plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
save_figure(fig, FINAL_OUT, "main_fig4_iso_main")
print("Saved →", FINAL_OUT / "main_fig4_iso_main.pdf")